# ❗Stretch Goals – SCD Type 2

## SCD Type 2 – dim_customer_scd2 (implemented)
**Source:** customer_changes.csv (25 events → 24 new versions; C0282 changes nothing and is ignored).
**Tracked attributes:** city, region, segment. **Result:** 525 rows (500 customers + 24 versions + unknown −1).

**Where the code is** (marked ❗):
- `02_silver`: `silver.customer_changes`
- `03_gold`: `dim_customer_scd2`, the `customer_scd2_key` column in `fact_sales` (added with a one-time `ALTER TABLE` migration) and the point-in-time join
- `04_validation`: SCD2 asserts
- `05_analytics`: Q3b (revenue by segment at time of purchase vs now) and Q3c (segment migration, also on the dashboard)

**Design decisions**
- Version 1 starts at 1900-01-01, not signup_date: 588 order lines are dated before signup (DQ log #14).
- Versions are rebuilt from the full change history on every run and MERGEd on (customer_id, valid_from):
  surrogate keys stay stable and re-runs change nothing. A daily incremental feed would instead need
  the "close current version + insert new version" pattern.
- valid_to = next valid_from − 1 day → periods never overlap, so the point-in-time join
  (order_date BETWEEN valid_from AND valid_to) matches exactly one version per line.
- fact_sales keeps both keys: customer_key (SCD1, customer as delivered in customers.csv) and
  customer_scd2_key (customer as they were on the order date).

**Proof:** 310 fact lines of 24 customers use version 2; silver net = gold net still holds (no duplication);
SCD2 asserts check one current version per customer and no overlapping periods.
